In [ ]:
# Perform all required imports
import polars as pl
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, auc, confusion_matrix
import matplotlib
matplotlib.use('Agg') # Force headless backend for mplib (needed for HPC)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import gc
import joblib
import os

In [ ]:
# Environment config
DATA_DIR = "./data/original-data/csv_files/train"
MODEL_PATH = "lgbm_baseline.joblib"
random_state = 42

In [ ]:
# Preprocess data files for model training, grouping all weeks into slices

try:
    df_base = pl.read_csv(f"{DATA_DIR}/train_base.csv")
    df_static_0 = pl.read_csv(f"{DATA_DIR}/train_static_0_0.csv")
    df_static_1 = pl.read_csv(f"{DATA_DIR}/train_static_0_1.csv")
except Exception as e:
    print(f"❌ Error loading data: {e}")
    # return None, None, None, None

# Join and Process
df_static = pl.concat([df_static_0, df_static_1], how="vertical_relaxed")
print("🔗 Joining Tables...")
df_train = df_base.join(df_static, on="case_id", how="left")
del df_static_0, df_static_1, df_static
gc.collect()

print("🧹 Preprocessing...")
null_counts = df_train.null_count()
total_rows = df_train.height
cols_to_keep = [col for col in df_train.columns if null_counts[col][0] / total_rows < 0.6]
df_train = df_train.select(cols_to_keep)

pdf_train = df_train.to_pandas()

cat_cols = pdf_train.select_dtypes(include=['object', 'string']).columns
for col in cat_cols:
    pdf_train[col] = pdf_train[col].astype('category')
    
X = pdf_train.drop(columns=['target', 'case_id', 'date_decision', 'WEEK_NUM'])
y = pdf_train['target']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=random_state, stratify=y)


In [ ]:
# Train LightGBM Model

if os.path.exists(MODEL_PATH):
    print(f"💾 Found saved model at '{MODEL_PATH}'. Loading...")
    model = joblib.load(MODEL_PATH)
    print("✅ Model loaded successfully!")
else:
    print(f"⚠️ No saved model found. Training new model...")
    model = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        random_state=random_state,
        n_jobs=-1 
    )
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)]
    )
    
    # SAVE THE MODEL IMMEDIATELY AFTER TRAINING
    print(f"💾 Saving model to '{MODEL_PATH}'...")
    joblib.dump(model, MODEL_PATH)

# --- STEP 3: SCORE & RETURN ---
# Even if we loaded from disk, we want to see the score on the current validation set
val_preds = model.predict_proba(X_val)[:, 1]
auc_score = roc_auc_score(y_val, val_preds)
print(f"\n✅ Baseline AUC Score: {auc_score:.4f}")